### A. Environment set up

In [1]:
# Auto realoading "magic" for IPython
# Load the autoreload extension
%load_ext autoreload 
# Automatically reload all modules (except those excluded by %aimport) every time before executing the Python code typed.
%autoreload 2 

In [ ]:
# Import necessary modules
import sys  # For manipulating the Python path
from pathlib import Path  # For handling filesystem paths in an object-oriented way
import pandas as pd  # For data manipulation and analysis
from sklearn.model_selection import train_test_split


# Add the src directory to the Python path to allow importing custom modules
from src.load_data import load_raw_data
from src.clean_data import clean_dataframe
from src.validate import validate_dataframe
from src.features import get_feature_preprocessor
from src.train import train_model

# If the notebook is in "notebooks/", the parent is the Project Root.
# Path.cwd() <- Current Working Directory
project_root = Path.cwd().parent

# This ensures Python looks here FIRST, before looking in installed packages.
if str(project_root) not in sys.path:
    # Convert Path object to string for sys.path
    sys.path.insert(0, str(project_root))
    print(f"Added to sys.path: {project_root}")

print("cwd:", Path.cwd())
print("project_root:", project_root)

Added to sys.path: /home/idiazl/2026_MLOps/2. mlops_modulirized-repo
cwd: /home/idiazl/2026_MLOps/2. mlops_modulirized-repo/notebooks
project_root: /home/idiazl/2026_MLOps/2. mlops_modulirized-repo


### B. Pipeline configuration

In [3]:
RAW_DATA_PATH = project_root / "data" / "raw" / "opiod_raw_data.csv"
TARGET_COLUMN = "OD"

print("Raw data path:", RAW_DATA_PATH)
print("Target column:", TARGET_COLUMN)

Raw data path: /home/idiazl/2026_MLOps/2. mlops_modulirized-repo/data/raw/opiod_raw_data.csv
Target column: OD


### 1. Execute load module

In [ ]:
df_raw = load_raw_data(RAW_DATA_PATH)
print("df_raw.shape:", df_raw.shape)

[load_data.load_raw_data] Loading raw data from /home/idiazl/2026_MLOps/2. mlops_modulirized-repo/data/raw/opiod_raw_data.csv
[utils.load_csv] Loading CSV from /home/idiazl/2026_MLOps/2. mlops_modulirized-repo/data/raw/opiod_raw_data.csv
[load_data.load_raw_data] Loaded dataframe shape: (1000, 22)
[clean_data.clean_dataframe] Cleaning dataframe
[clean_data.clean_dataframe] Dropped 102 rows due to NA or duplicates
[clean_data.clean_dataframe] Rows after cleaning: 898
df_raw.shape: (1000, 22)
df_clean.shape: (898, 21)


### 2. Execute clean module

In [ ]:
df_clean = clean_dataframe(df_raw, target_column=TARGET_COLUMN)
print("df_clean.shape:", df_clean.shape)

### --- quick verfication ---

In [5]:
# 1) Column normalization proof
raw_has_rx_ds = "rx ds" in df_raw.columns
clean_has_rx_ds = "rx_ds" in df_clean.columns

print('Raw has "rx ds":', raw_has_rx_ds)
print('Clean has "rx_ds":', clean_has_rx_ds)

if raw_has_rx_ds:
    assert clean_has_rx_ds, 'Expected "rx ds" to be normalized to "rx_ds"'

# 2) Index integrity proof
index_is_contiguous = df_clean.index.equals(pd.RangeIndex(len(df_clean)))
print("Index reset and contiguous:", index_is_contiguous)
assert index_is_contiguous, "Expected RangeIndex(0..n-1) after reset_index(drop=True)"

# 3) Target presence proof
target_present = TARGET_COLUMN in df_clean.columns
print(f"Target '{TARGET_COLUMN}' preserved:", target_present)
assert target_present, f"Missing target column after cleaning: {TARGET_COLUMN}"

df_clean.head()

Raw has "rx ds": True
Clean has "rx_ds": True
Index reset and contiguous: True
Target 'OD' preserved: True


,OD,Low_inc,SURG,rx_ds,A,B,C,D,E,F,...,I,J,K,L,M,N,R,S,T,V
0,1,1,0,330,0,0,0,1,1,1,...,1,1,0,1,1,1,1,0,0,0
1,1,1,0,457,0,0,1,0,1,0,...,1,1,0,0,1,1,1,1,0,0
2,1,1,1,722,0,1,0,1,1,0,...,1,1,1,0,1,0,1,0,1,0
3,0,1,0,262,0,1,0,0,1,1,...,1,0,1,1,1,1,1,1,1,0
4,1,1,0,780,0,0,0,0,1,1,...,1,0,0,0,1,0,1,0,0,0


In [6]:
# Column Comparison (first 20)

raw_cols = list(df_raw.columns)
clean_cols = list(df_clean.columns)

print("\nRaw columns:")
print(raw_cols[:20])

print("\nClean columns:")
print(clean_cols[:20])

# Optional structured diff for teaching
removed_cols = sorted(set(raw_cols) - set(clean_cols))
added_cols = sorted(set(clean_cols) - set(raw_cols))

print("\nColumns removed during cleaning:")
print(removed_cols[:10])

print("\nColumns added during cleaning:")
print(added_cols[:10])


Raw columns:
['ID', 'OD', 'Low_inc', 'SURG', 'rx ds', 'A', 'B', 'C', 'D', 'E', 'F', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'R', 'S']

Clean columns:
['OD', 'Low_inc', 'SURG', 'rx_ds', 'A', 'B', 'C', 'D', 'E', 'F', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'R', 'S', 'T']

Columns removed during cleaning:
['ID', 'rx ds']

Columns added during cleaning:
['rx_ds']


### 3. Data Validation (Security Gate)
**Goal:** Fail fast on broken assumptions before feature engineering or training

In [7]:
# Validation settings and required columns
TARGET_COLUMN = "OD"
BINARY_SUM_COLS = ["A","B","C","D","E","F","H","I","J","K","L","M","N","R","S","T","Low_inc","SURG"]

VALIDATION_SETTINGS = {
    "required_columns": {
        "quantile_bin": ["rx_ds"],
        "binary_sum_cols": BINARY_SUM_COLS,
        "categorical_onehot": [],
        "numeric_passthrough": [],
    },
    "target_allowed_values": [0, 1],        # Demo rule for this dataset only
    "numeric_non_negative_cols": ["rx_ds"], # Demo rule for this dataset only
    "check_missing_values": True,
}

required_columns = (
    [TARGET_COLUMN]
    + VALIDATION_SETTINGS["required_columns"]["quantile_bin"]
    + VALIDATION_SETTINGS["required_columns"]["categorical_onehot"]
    + VALIDATION_SETTINGS["required_columns"]["numeric_passthrough"]
    + VALIDATION_SETTINGS["required_columns"]["binary_sum_cols"]
)

print(f"[notebook] df_clean shape: {df_clean.shape}")
print(f"[notebook] Validating required columns: {required_columns}")

[notebook] df_clean shape: (898, 21)
[notebook] Validating required columns: ['OD', 'rx_ds', 'A', 'B', 'C', 'D', 'E', 'F', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'R', 'S', 'T', 'Low_inc', 'SURG']


In [8]:
validate_dataframe(
    df=df_clean,
    required_columns=required_columns,
    check_missing_values=VALIDATION_SETTINGS["check_missing_values"],
    target_column=TARGET_COLUMN,
    target_allowed_values=VALIDATION_SETTINGS["target_allowed_values"],
    numeric_non_negative_cols=VALIDATION_SETTINGS["numeric_non_negative_cols"],
)

print("[notebook] Validation passed, safe to proceed")

[validate.validate_dataframe] Validating dataframe
[notebook] Validation passed, safe to proceed


### 4. Feature engineering (Recipe)

In [ ]:
# Build the recipe only
# This is a blueprint, it does not learn anything until we call pipeline.fit(X_train, y_train):
# - Imputation (handles missing values safely at inference time)
# - Quantile binning as ordinal codes (0..n_bins-1), then scaling
# - Binary sum feature (row-wise sum of indicator columns), then scaling
# - One hot encoding for categoricals (robust to unseen categories)
preprocessor = get_feature_preprocessor(
    quantile_bin_cols=FEATURE_SETTINGS["quantile_bin"],
    categorical_onehot_cols=FEATURE_SETTINGS["categorical_onehot"],
    numeric_passthrough_cols=FEATURE_SETTINGS["numeric_passthrough"],
    binary_sum_cols=FEATURE_SETTINGS["binary_sum_cols"],
    n_bins=FEATURE_SETTINGS["n_bins"],
)

print("[notebook] Feature recipe built (not fitted yet)")
preprocessor

### 5. raining - Three-way split (Train, Validation, Test)
- Train (80%) learns feature rules and model weights
- Validation (15%) checks performance during development
- Test (5%) stays untouched as an audit vault
We will validate on the validation set, and simulate inference using 10 rows from the test vault

In [ ]:
TARGET_COLUMN = "OD"
RANDOM_STATE = 42

X = df_clean.drop(columns=[TARGET_COLUMN])
y = df_clean[TARGET_COLUMN]

TEST_SIZE = 0.05
VAL_SIZE = 0.15

# Step A: carve out the test vault first
# This keeps the test set untouched until the end
X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

# Step B: split remaining into train and validation
# We want VAL_SIZE of the total dataset, taken from the remaining (1 - TEST_SIZE)
relative_val_size = VAL_SIZE / (1.0 - TEST_SIZE)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=relative_val_size,
    random_state=RANDOM_STATE,
    stratify=y_temp,
)

print("[notebook] 3-way split complete")
print("Train:", X_train.shape[0], "Validation:", X_val.shape[0], "Test:", X_test.shape[0])

In [ ]:
# Training fits the recipe and the model on X_train only
model_pipeline = train_model(
    X_train=X_train,
    y_train=y_train,
    preprocessor=preprocessor,
    problem_type="classification",
)

# Validation predictions used for evaluation in the next step
val_preds = model_pipeline.predict(X_val)

print("[notebook] Training complete")
print("[notebook] Validation predictions:", len(val_preds))

In [ ]:
# Simulate a real inference request
# We sample 10 rows from the untouched test vault
X_infer = X_test.sample(n=10, random_state=RANDOM_STATE)
infer_preds = model_pipeline.predict(X_infer)

print("[notebook] Inference predictions:", infer_preds.tolist())